# SentinelX — Exploratory Data Analysis & Fraud Model Evaluation

**Project:** SentinelX — Intelligent Fraud Detection & Transaction Risk Intelligence Platform  
**Dataset:** PaySim Synthetic Financial Mobile Money Transactions  
**Author:** FinTech Risk Engineering Team  

---

### Objectives:
1. **Exploratory Data Analysis (EDA):** Analyze transaction types, volume distributions, and severe class imbalance.
2. **Feature Engineering & Leakage Prevention:** Derive mathematical balance error checks (`errorBalanceOrig`, `errorBalanceDest`, `balance_drained`) while strictly eliminating pre-existing flags (`isFlaggedFraud`).
3. **Model Development & Benchmarking:** Compare Logistic Regression (Linear Baseline), Random Forest (Non-linear Baseline), and XGBoost (Production Candidate).
4. **Class Imbalance Metrics:** Measure PR-AUC, ROC-AUC, Recall, Precision, and False Positives per 1,000 transactions.
5. **Explainable AI (Tree SHAP):** Extract exact Shapley feature contributions for suspicious transactions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('.')))
if project_root not in sys.path:
    sys.path.append(project_root)

from ml.data.dataset_generator import generate_synthetic_paysim
from ml.feature_engineering.engineer import engineer_features, prepare_train_test_split
from ml.evaluation.metrics import evaluate_fraud_model

print("Libraries and SentinelX pipeline modules imported successfully.")

## 1. Dataset Generation & EDA

We generate 15,000 synthetic PaySim transactions simulating mobile money streams with known money mule cash-outs, balance draining, and normal payment behavior.

In [ ]:
df = generate_synthetic_paysim(n_samples=15000, fraud_ratio=0.05, random_state=42)
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
print(df['isFraud'].value_counts(normalize=True).rename({0: 'Legitimate', 1: 'Fraudulent'}))
df.head()

### Transaction Type vs Fraud Occurrence
In PaySim, fraud is strictly confined to `TRANSFER` (account takeover / fund diversion) and `CASH_OUT` (money mule cashing out at an agent).

In [ ]:
pd.crosstab(df['type'], df['isFraud'], margins=True)

## 2. Feature Engineering & Label Leakage Shield

In standard transactions:
$$\text{newbalanceOrig} = \text{oldbalanceOrg} - \text{amount} \implies \text{errorBalanceOrig} = \text{newbalanceOrig} + \text{amount} - \text{oldbalanceOrg} = 0$$

When an unauthorized account drainage occurs, balance errors deviate sharply.
**Leakage Elimination:** `isFlaggedFraud` is explicitly dropped to prevent the model from learning rule shortcuts.

In [ ]:
X_train, X_test, y_train, y_test = prepare_train_test_split(df, test_size=0.25, chronological=True)
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print("Engineered Feature Columns:", list(X_train.columns))
assert 'isFlaggedFraud' not in X_train.columns, "Leakage detected!"

## 3. Model Training & Imbalance Calibration

We evaluate 3 models:
1. **Logistic Regression** (Linear baseline with class_weight='balanced')
2. **Random Forest** (Nonlinear tree ensemble)
3. **XGBoost** (Production candidate with cost-sensitive `scale_pos_weight`)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Scale for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]

# 3. XGBoost
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, scale_pos_weight=scale_pos, eval_metric='aucpr', random_state=42)
xgb.fit(X_train, y_train)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("All 3 models trained successfully.")

## 4. Rigorous Benchmark Evaluation

We measure Precision, Recall, F1, PR-AUC, ROC-AUC, and False Positives per 1,000 transactions.

In [ ]:
results = {
    'Logistic Regression': evaluate_fraud_model(y_test.values, lr_prob),
    'Random Forest': evaluate_fraud_model(y_test.values, rf_prob),
    'XGBoost (Production)': evaluate_fraud_model(y_test.values, xgb_prob)
}

eval_df = pd.DataFrame(results).T[['precision', 'recall', 'f1_score', 'pr_auc', 'roc_auc', 'fp_per_1000']]
eval_df

## 5. Explainable AI with Tree SHAP

We extract exact Shapley feature values for high-risk suspicious transactions.

In [ ]:
from ml.explainability.shap_explainer import SentinelXExplainer, FEATURE_COLUMNS

explainer = SentinelXExplainer(xgb, FEATURE_COLUMNS)
sample_fraud = X_test[y_test == 1].iloc[[0]]
explanations = explainer.explain_transaction(sample_fraud)

print("Forensic Tree SHAP Impact for Sample Fraud Transaction:")
for exp in explanations[:5]:
    print(f"  • {exp['label']:35s} | SHAP: {exp['shap_value']:+.4f} | {exp['summary']}")